# 1 – Indexing: PDF-Dokumente in eine Vektordatenbank überführen

Dieses Notebook führt die **Indexing-Pipeline** durch:

1. **Laden** – PDFs und Word-Dokumente werden eingelesen
2. **Chunking** – Der Text wird in überlappende Abschnitte zerlegt
3. **Embedding** – Jeder Chunk wird in einen Vektor umgewandelt
4. **Speichern** – Die Vektoren landen in einer lokalen ChromaDB

> **Hinweis:** Dieses Notebook muss nur einmal (oder bei neuen Dokumenten) ausgeführt werden.  
> Das RAG-Notebook (Notebook 2) liest dann die fertige Datenbank.

## Imports & Konfiguration

In [3]:
import os
import glob
import shutil

from langchain_community.document_loaders import PyMuPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

d:\05_VSCode\05_Python\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# --- KONFIGURATION ---

# Pfade
PDF_SOURCE_DIR = "./documents"      # Ordner mit den Quell-Dokumenten (PDF + DOCX)
DB_DIR         = "./chroma_db"       # Speicherort der Vektordatenbank

# LM Studio – läuft lokal und stellt eine OpenAI-kompatible API bereit
LM_STUDIO_URL  = "http://127.0.0.1:1234/v1"
EMBEDDING_MODEL = "text-embedding-multilingual-e5-large-instruct"  # Modellname wie in LM Studio geladen

# Chunking-Parameter
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 128   # ~25 % Überlappung – wichtig bei langen deutschen Sätzen

## Embedding-Modell vorbereiten (via LM Studio)

Das Modell `multilingual-e5-large-instruct` läuft in **LM Studio** auf der GPU  
und wird über eine OpenAI-kompatible API angesprochen (`http://localhost:1234/v1`).

Das Modell erwartet **spezifische Präfixe**:  
- Dokumente (beim Indexieren): `"passage: ..."`  
- Suchanfragen (beim Retrieval): `"query: ..."`  

Weder LangChain noch LM Studio setzen diese automatisch, deshalb der Wrapper:

In [5]:
class E5Embeddings(OpenAIEmbeddings):
    """Wrapper, der die vom E5-Modell erwarteten Präfixe automatisch setzt."""

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return super().embed_documents(["passage: " + t for t in texts])

    def embed_query(self, text: str) -> list[float]:
        return super().embed_query("query: " + text)


embeddings = E5Embeddings(
    model=EMBEDDING_MODEL,
    api_key="lm-studio",   # LM Studio braucht keinen echten Key
    base_url=LM_STUDIO_URL,
    check_embedding_ctx_length=False,
)

## Schritt 1 – Dokumente laden (PDF & DOCX)

In [6]:
# Mapping: Dateiendung → passender LangChain-Loader
LOADERS = {
    ".pdf":  PyMuPDFLoader,
    ".docx": Docx2txtLoader,
}


def load_documents(source_dir: str):
    """Lädt alle unterstützten Dokumente (PDF, DOCX) aus einem Verzeichnis."""
    all_docs = []
    file_count = 0

    for ext, loader_cls in LOADERS.items():
        for filepath in glob.glob(os.path.join(source_dir, f"*{ext}")):
            loader = loader_cls(filepath)
            all_docs.extend(loader.load())
            print(f"  📄 {os.path.basename(filepath)}")
            file_count += 1

    print(f"\n✅ {len(all_docs)} Seiten/Abschnitte aus {file_count} Dokumenten geladen.")
    return all_docs

## Schritt 2 – Text in Chunks aufteilen

In [7]:
def split_documents(docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """Zerlegt Dokumente in überlappende Text-Chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = splitter.split_documents(docs)
    print(f"✅ {len(chunks)} Chunks erzeugt (Größe: {chunk_size}, Overlap: {chunk_overlap})")
    return chunks

## Schritt 3 – Embeddings erstellen & in ChromaDB speichern

In [8]:
def create_vectorstore(chunks, embedding_fn, db_dir=DB_DIR):
    """Erzeugt Embeddings und speichert sie in einer lokalen ChromaDB."""
    # Alten Index löschen, damit keine veralteten Vektoren drin bleiben
    if os.path.exists(db_dir):
        shutil.rmtree(db_dir)
        print("🗑️  Alter Index gelöscht.")

    print("🧠 Erstelle Embeddings und speichere in ChromaDB...")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_fn,
        persist_directory=db_dir,
    )
    print(f"✨ Fertig! Datenbank gespeichert in '{db_dir}'")
    return vectorstore

## Pipeline ausführen

In [9]:
# --- PIPELINE ---
print("🚀 Starte Indexing-Pipeline...\n")

docs   = load_documents(PDF_SOURCE_DIR)
chunks = split_documents(docs)
db     = create_vectorstore(chunks, embeddings)

print(f"\n🎯 Nächster Schritt: Notebook 2 (RAG) öffnen und Fragen stellen.")

🚀 Starte Indexing-Pipeline...

  📄 20240604-Shortpaper_Standards-fuer-KI.pdf
  📄 2104.05314v2.pdf
  📄 C_2025_884_2_EN_ACT_part1_v2_OlFHGWKJRcz1XPvgNW2ZVuggMw_112366.pdf
  📄 EU-6-001-25-pdf.pdf
  📄 Guidelines_on_prohibited_artificial_intelligence_practices_established_by_Regulation_EU_20241689_AI_Act_German.PDF
  📄 gutachten-datenethikkommission.pdf
  📄 Künstliche Intelligenz im Rahmen des AI Acts der EU.pdf
  📄 OJ_L_202401689_DE_TXT.pdf
  📄 Regulierung von KI _ Künstliche Intelligenz _ bpb.de.pdf

✅ 651 Seiten/Abschnitte aus 9 Dokumenten geladen.
✅ 5961 Chunks erzeugt (Größe: 512, Overlap: 128)
🗑️  Alter Index gelöscht.
🧠 Erstelle Embeddings und speichere in ChromaDB...
✨ Fertig! Datenbank gespeichert in './chroma_db'

🎯 Nächster Schritt: Notebook 2 (RAG) öffnen und Fragen stellen.


In [ ]:
import requests 
r = requests.get("http://localhost:1234/v1/models") 
print(r.json())

In [ ]:
import requests 
r = requests.post( 
    "http://localhost:1234/v1/embeddings", 
    json={"model": "text-embedding-multilingual-e5-large-instruct", "input": "Testtext"} 
    ) 
print(r.status_code, r.json())
